In [ ]:
from matchms import Spectrum
import numpy as np


def attention(query:np.array, reference:np.array):
    dot_product = np.dot(query, reference)
    softmax_weights = np.exp(dot_product) / np.sum(np.exp(dot_product))
    return softmax_weights

In [4]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

11499
2253216


In [5]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset, collate_fun, collate_fun_emb
import numpy as np


dataset_lib = SpecDataset(mols_all)
dataset_test = SpecDataset(mols_test)
loader_lib = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
loader_test = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)

In [6]:
import torch.optim as optim
from utils.model import Spec2Emb, Linear_Scheduler


gpu = 8
model = Spec2Emb().to(gpu)
model.load_state_dict(pt.load('./model/base_peak0.01_epoch4.pth'))

<All keys matched successfully>

In [7]:
from utils.tools import gen_embeddings, build_idx, evaluate

embeddings_lib = gen_embeddings(model, loader_lib, gpu, power=0.4) 
embeddings_test = gen_embeddings(model, loader_test, gpu, power=0.4)
I_expand, _ = build_idx(embeddings_lib, embeddings_test, gpu, topk=200)
top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, library_type='expanded')
I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, gpu, topk=200)
top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, library_type='insilico')

Searching time:  0:00:01.621238
expanded library
Top1 hit rate: 40.33%
Top10 hit rate: 81.27%
Searching time:  0:00:01.525386
insilico library
Top1 hit rate: 40.65%
Top10 hit rate: 81.62%


In [47]:
max_indices = []

(11499, 200)


In [12]:
cnt_1 = 0
cnt_10 = 0
for i in range(11499):
    simi = [attention(embeddings_test[i], embeddings_lib[I_insilico[i][j]]) for j in range(100)]
    simi_idx = [(simi[j], I_insilico[i][j]) for j in range(100)]
    simi_idx.sort(key=lambda x: x[0], reverse=True)
    if mols_test[i].metadata['inchikey'] == mols_all[simi_idx[0][1]].metadata['inchikey']:
        cnt_1 += 1
    for j in range(10):
        if mols_test[i].metadata['inchikey'] == mols_all[simi_idx[j][1]].metadata['inchikey']:
            cnt_10 += 1
            break
        
print(cnt_1/11499)
print(cnt_10/11499)

0.4064701278372032
0.8162448908600748


In [9]:
pt.cuda.empty_cache()